# Proyecto integrador · Sesión 5 — Informe de PQRS y aportes para la Dirección
**PY-EAN · Python para Procesamiento, Calidad y Automatización de Datos** · Universidad Ean

**Caso:** la Dirección del fondo de pensiones (ficticio) pide un informe mensual con tres preguntas:
1. ¿Estamos respondiendo las **PQRS** a tiempo (15 días hábiles)?
2. ¿Cómo va el **recaudo de aportes** y qué afiliados activos llevan **3 o más meses sin aportar**?
3. ¿Qué tan **confiables** son los datos con los que respondemos?

Tu misión: construir un **pipeline** que responda todo con un solo `main()` y entregue un Excel.

| Parte | Etapa | Tiempo |
|---|---|---|
| 0 | Configuración | 3 min |
| 1 | Ingestar: cargar todo (incluidos los 9 meses de aportes) | 10 min |
| 2 | Procesar: limpiar (función dada) | 5 min |
| 3 | Validar: 8 reglas de calidad | 12 min |
| 4 | Indicadores de PQRS | 10 min |
| 5 | Indicadores de aportes y mora | 12 min |
| 6 | Proteger datos personales | 5 min |
| 7 | Exportar el informe y `main()` | 8 min |

Donde veas `✏️` completa el código. Puedes trabajar en equipo (2–3 personas).

> Todos los datos son sintéticos. No corresponden a personas reales.

## Parte 0 · Configuración

In [ ]:
# ▶️ Ejecuta esta celda primero (Shift + Enter)
import os
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/Hackwy402/python-procesamiento-ean/main/datos/"
# Si subiste la carpeta "datos" a Colab (o trabajas en tu PC) se usa la copia local;
# si no, se leen los datos directamente desde GitHub.
for carpeta in ["datos/", "../datos/", "/content/datos/"]:
    if os.path.exists(carpeta + "afiliados.csv"):
        RUTA = carpeta
        break
else:
    RUTA = URL_DATOS

pd.set_option("display.max_columns", 30)
print("Leyendo datos desde:", RUTA)

In [ ]:
from datetime import date
import hashlib
import matplotlib.pyplot as plt

CONFIG = {
    "SMMLV": 1_750_905,             # salario mínimo 2026 (dato de referencia del curso)
    "TASA_APORTE": 0.16,
    "DIAS_HABILES_RESPUESTA": 15,   # término general de respuesta a peticiones
    "MESES": range(1, 10),          # enero a septiembre de 2026
    "MESES_MORA": 3,
    "CARPETA_SALIDA": "salidas/",
    "SAL": "ean-curso-2026",        # en la vida real: fuera del código
}
# Festivos de Colombia 2026 (verificar contra el calendario oficial)
FESTIVOS_2026 = ["2026-01-01", "2026-01-12", "2026-03-23", "2026-04-02", "2026-04-03",
                 "2026-05-01", "2026-05-18", "2026-06-08", "2026-06-15", "2026-06-29",
                 "2026-07-20", "2026-08-07", "2026-08-17", "2026-10-12", "2026-11-02",
                 "2026-11-16", "2026-12-08", "2026-12-25"]
os.makedirs(CONFIG["CARPETA_SALIDA"], exist_ok=True)

def log(mensaje):
    """Bitácora mínima: hora + mensaje."""
    print(f"[{pd.Timestamp.now():%H:%M:%S}] {mensaje}")

log("Configuración lista")

## Parte 1 · Ingestar
Los aportes llegan en **un archivo por mes**: `aportes/aportes_2026-01.csv` … `aportes_2026-09.csv`.

⚠️ Recuerda la Sesión 4: **julio llegó distinto** (separador `;` y encabezados en MAYÚSCULAS).

**✏️ Ejercicio 1.** Completa `cargar_aportes()`:
- recorre `CONFIG["MESES"]` y arma el nombre de cada archivo,
- léelo con `pd.read_csv(archivo, sep=None, engine="python")` (detecta el separador solo),
- pasa los nombres de columnas a minúscula,
- une todo con `pd.concat`.

In [ ]:
def cargar_aportes(ruta=RUTA):
    partes = []
    for mes in CONFIG["MESES"]:
        archivo = f"{ruta}aportes/aportes_2026-{mes:02d}.csv"
        # ✏️ 1) lee el archivo detectando el separador
        # ✏️ 2) pasa las columnas a minúscula
        # ✏️ 3) agrega el DataFrame a la lista partes
        ...
    # ✏️ 4) devuelve todas las partes unidas con pd.concat
    ...

In [ ]:
def cargar_datos(ruta=RUTA):
    afiliados = pd.read_csv(ruta + "afiliados.csv", dtype={"numero_documento": str, "telefono": str})
    pqrs = pd.read_csv(ruta + "pqrs.csv", parse_dates=["fecha_radicacion", "fecha_respuesta"])
    catalogo = pd.read_excel(ruta + "oficinas.xlsx", sheet_name="ciudades")
    aportes = cargar_aportes(ruta)
    log(f"Cargados: {len(afiliados):,} afiliados · {len(pqrs):,} PQRS · {len(aportes):,} aportes")
    return afiliados, pqrs, aportes, catalogo

afiliados_raw, pqrs, aportes_raw, catalogo = cargar_datos()
aportes_raw.head()

## Parte 2 · Procesar (función dada)
Para no repetir la Sesión 2, la limpieza de afiliados ya está hecha. **Léela** y ejecútala: identifica cada
transformación (documentos, nombres, género, estado, ciudad, fechas, salario).

In [ ]:
import re
import unicodedata
import numpy as np

def normalizar_texto(s):
    """minúsculas, sin tildes, sin puntos, espacios simples."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace(".", "")
    return re.sub(r"\s+", " ", s).strip()

ALIAS_CIUDAD = {"bogota": "bogota dc", "b/quilla": "barranquilla", "b/manga": "bucaramanga",
                "santiago de cali": "cali", "cartagena de indias": "cartagena"}

def a_fecha(serie):
    """Acepta AAAA-MM-DD y DD/MM/AAAA; fechas imposibles → NaT."""
    f = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    f = f.fillna(pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce"))
    return f.where((f >= "1920-01-01") & (f <= pd.Timestamp.today()))

def limpiar_afiliados(df, catalogo):
    """Estandariza afiliados (lo que hicimos en la Sesión 2). NO elimina documentos repetidos:
    eso lo mide la regla de calidad y lo depura luego depurar_afiliados()."""
    df = df.drop_duplicates().copy()
    texto = ["tipo_documento", "numero_documento", "nombres", "apellidos", "genero",
             "ciudad", "departamento", "email", "estado"]
    df[texto] = df[texto].apply(lambda col: col.str.strip())
    df["numero_documento"] = df["numero_documento"].str.replace(r"[^0-9]", "", regex=True)
    df["nombres"] = df["nombres"].str.title()
    df["apellidos"] = df["apellidos"].str.title()
    df["genero"] = df["genero"].str.upper().str[0]
    df["estado"] = df["estado"].str.capitalize().replace({"Pensionada": "Pensionado"})
    canon = {normalizar_texto(c): c for c in catalogo["ciudad"]}
    norm = df["ciudad"].map(normalizar_texto).replace(ALIAS_CIUDAD)
    df["ciudad"] = norm.map(canon).fillna(df["ciudad"])
    df["fecha_nacimiento"] = a_fecha(df["fecha_nacimiento"])
    df["fecha_afiliacion"] = a_fecha(df["fecha_afiliacion"])
    sal = pd.to_numeric(df["salario_base"].astype(str).str.replace(".", "", regex=False), errors="coerce")
    df["salario_base"] = sal.where(sal > 0)
    df["email"] = df["email"].str.lower()
    df["telefono"] = df["telefono"].astype("string").str.replace(r"\.0$", "", regex=True)
    return df

def depurar_afiliados(df):
    """Un registro por documento (se queda con el id más antiguo)."""
    con_doc = df[df["numero_documento"].notna() & (df["numero_documento"] != "")]
    return con_doc.sort_values("id_afiliado").drop_duplicates("numero_documento")

In [ ]:
def limpiar_aportes(df):
    df = df.drop_duplicates().copy()
    df["fecha_pago"] = pd.to_datetime(df["fecha_pago"], errors="coerce")
    return df

afiliados = limpiar_afiliados(afiliados_raw, catalogo)
aportes = limpiar_aportes(aportes_raw)
log(f"Afiliados: {len(afiliados_raw):,} → {len(afiliados):,} tras quitar duplicados exactos")
log(f"Aportes:   {len(aportes_raw):,} → {len(aportes):,} tras quitar duplicados exactos")
afiliados["estado"].value_counts()

## Parte 3 · Validar: 8 reglas de calidad
Cada regla es una **máscara booleana**: `True` = el registro cumple. La función `regla()` calcula el cumplimiento.

**✏️ Ejercicio 2.** Completa las reglas 3, 4, 6 y 7 (las otras ya están como ejemplo).

In [ ]:
def regla(n, tabla, nombre, dimension, cumple):
    total = len(cumple)
    ok = int(cumple.sum())
    return {"#": n, "tabla": tabla, "regla": nombre, "dimensión": dimension,
            "registros": total, "incumplen": total - ok, "% cumplimiento": round(ok / total * 100, 2)}

def evaluar_calidad(af, ap, pq, catalogo):
    reglas = [
        regla(1, "afiliados", "Documento con 6 a 10 dígitos", "Validez",
              af["numero_documento"].fillna("").str.fullmatch(r"\d{6,10}")),
        regla(2, "afiliados", "Documento único", "Unicidad",
              ~af["numero_documento"].duplicated(keep=False) & af["numero_documento"].notna()),
        regla(3, "afiliados", "Salario ≥ SMMLV", "Validez",
              ...),   # ✏️ salario_base mayor o igual a CONFIG["SMMLV"]
        regla(4, "afiliados", "Email con formato válido", "Validez",
              ...),   # ✏️ pista: .fillna("").str.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.]+")
        regla(5, "afiliados", "Ciudad en catálogo", "Consistencia",
              af["ciudad"].isin(catalogo["ciudad"])),
        regla(6, "aportes", "Aporte = 16 % del IBC (±1 peso)", "Exactitud",
              ...),   # ✏️ |valor_aporte − ibc × TASA_APORTE| <= 1   (usa .abs())
        regla(7, "aportes", "El afiliado existe", "Integridad",
              ...),   # ✏️ id_afiliado de aportes está en id_afiliado de afiliados (.isin)
        regla(8, "pqrs", "Fechas coherentes (cerrada con respuesta ≥ radicación)", "Consistencia",
              ~((pq["estado"] == "Cerrado") & pq["fecha_respuesta"].isna())
              & ~(pq["fecha_respuesta"] < pq["fecha_radicacion"])),
    ]
    return pd.DataFrame(reglas)

calidad = evaluar_calidad(afiliados, aportes, pqrs, catalogo)
calidad

✍️ **Interpreta:** ¿cuál regla tiene el peor cumplimiento? ¿Es grave para el informe de la Dirección? ¿Por qué?

## Parte 4 · Indicadores de PQRS
**✏️ Ejercicio 3.** Completa `indicadores_pqrs()`:
- calcula `dias_habiles` entre radicación y respuesta con `np.busday_count` (solo PQRS **cerradas** con fechas válidas),
- `a_tiempo` = `dias_habiles <= 15`,
- resume por **tipo**, **tema** y **canal**.

In [ ]:
def indicadores_pqrs(pq):
    pq = pq.copy()
    cerradas = pq[(pq["estado"] == "Cerrado") & pq["fecha_respuesta"].notna()
                  & (pq["fecha_respuesta"] >= pq["fecha_radicacion"])].copy()
    inicio = cerradas["fecha_radicacion"].values.astype("datetime64[D]")
    fin = cerradas["fecha_respuesta"].values.astype("datetime64[D]")
    cerradas["dias_habiles"] = ...   # ✏️ np.busday_count(inicio, fin, holidays=FESTIVOS_2026)
    cerradas["a_tiempo"] = ...       # ✏️ dias_habiles <= CONFIG["DIAS_HABILES_RESPUESTA"]

    def resumen(col):
        total = pq.groupby(col).size().rename("radicadas")
        oport = cerradas.groupby(col).agg(cerradas_validas=("radicado", "count"),
                                          dias_promedio=("dias_habiles", "mean"),
                                          pct_a_tiempo=("a_tiempo", "mean"))
        out = pd.concat([total, oport], axis=1).reset_index()
        out["dias_promedio"] = out["dias_promedio"].round(1)
        out["pct_a_tiempo"] = (out["pct_a_tiempo"] * 100).round(1)
        return out.sort_values("radicadas", ascending=False)

    kpis = {
        "PQRS radicadas": len(pq),
        "PQRS abiertas o en trámite": ...,                 # ✏️ estado en ["Abierto", "En trámite"]
        "Días hábiles promedio de respuesta": ...,         # ✏️ promedio de dias_habiles (1 decimal)
        "% respondidas dentro de 15 días hábiles": ...,    # ✏️ promedio de a_tiempo × 100 (1 decimal)
    }
    return {"kpis": kpis, "por_tipo": resumen("tipo"), "por_tema": resumen("tema"),
            "por_canal": resumen("canal"), "detalle": cerradas}

kpi_pqrs = indicadores_pqrs(pqrs)
display(kpi_pqrs["kpis"])
kpi_pqrs["por_tema"]

## Parte 5 · Indicadores de aportes y mora
**✏️ Ejercicio 4.** Completa `indicadores_aportes()`:
- **recaudo mensual**: suma de `valor_aporte` y aportantes únicos por `periodo`,
- **por regional**: une aportes → afiliados (ciudad) → catálogo (regional) con `merge`,
- **mora**: afiliados **activos** (depurados) con **3 o más** de los 9 meses sin aporte.

In [ ]:
def indicadores_aportes(ap, af, catalogo):
    validos = ap[ap["id_afiliado"].isin(af["id_afiliado"])]   # solo aportes con afiliado existente

    # ✏️ recaudo mensual: groupby("periodo") con suma de valor_aporte y nunique de id_afiliado
    mensual = ...

    # ✏️ por regional: validos → merge con af[["id_afiliado","ciudad"]] → merge con catalogo[["ciudad","regional"]]
    con_regional = ...
    por_regional = (con_regional.groupby("regional")
                    .agg(recaudo_total=("valor_aporte", "sum"), aportantes=("id_afiliado", "nunique"))
                    .sort_values("recaudo_total", ascending=False).reset_index())

    activos = depurar_afiliados(af)
    activos = activos[activos["estado"] == "Activo"]
    meses_con_aporte = validos.groupby("id_afiliado")["periodo"].nunique()
    activos = activos.assign(meses_con_aporte=activos["id_afiliado"].map(meses_con_aporte).fillna(0).astype(int))
    activos["meses_sin_aporte"] = len(CONFIG["MESES"]) - activos["meses_con_aporte"]
    mora = ...   # ✏️ activos con meses_sin_aporte >= CONFIG["MESES_MORA"]
    mora = mora.merge(catalogo[["ciudad", "regional"]], on="ciudad", how="left")
    mora = mora[["id_afiliado", "numero_documento", "nombres", "apellidos", "ciudad", "regional",
                 "meses_con_aporte", "meses_sin_aporte"]].sort_values("meses_sin_aporte", ascending=False)

    kpis = {
        "Recaudo total ene–sep 2026": int(mensual["recaudo"].sum()),
        "Recaudo promedio mensual": int(mensual["recaudo"].mean()),
        "Aportantes promedio por mes": int(mensual["aportantes"].mean()),
        "Afiliados activos (depurados)": len(activos),
        "Activos con 3+ meses sin aportar": len(mora),
    }
    return {"kpis": kpis, "mensual": mensual, "por_regional": por_regional, "mora": mora}

kpi_aport = indicadores_aportes(aportes, afiliados, catalogo)
display(kpi_aport["kpis"])
display(kpi_aport["mensual"])
kpi_aport["por_regional"]

## Parte 6 · Proteger datos personales
La lista de mora va para un área operativa: **no** debe llevar cédula ni nombre.

**✏️ Ejercicio 5.** Completa `seudonimizar()`: crea `doc_seudo` con SHA-256 (sal + documento, primeros 12 caracteres)
y elimina `numero_documento`, `nombres` y `apellidos`.

In [ ]:
def seudonimizar_doc(documento):
    texto = CONFIG["SAL"] + str(documento)
    return ...   # ✏️ hashlib.sha256(texto.encode("utf-8")).hexdigest()[:12]

def seudonimizar(df):
    df = df.copy()
    df.insert(1, "doc_seudo", ...)   # ✏️ aplica seudonimizar_doc a numero_documento con .map
    return ...                       # ✏️ elimina numero_documento, nombres y apellidos

mora_segura = seudonimizar(kpi_aport["mora"])
print("¿Mismo documento → mismo código?", seudonimizar_doc("52345678") == seudonimizar_doc("52345678"))
mora_segura.head()

## Parte 7 · Exportar el informe y `main()`
La función `graficos()` ya está lista. **✏️ Ejercicio 6.** Completa `exportar_informe()` escribiendo cada hoja.
**✏️ Ejercicio 7.** Ejecuta `main()` y descarga tu informe.

In [ ]:
def graficos(kpi_pqrs, kpi_aport, carpeta):
    rutas = []
    fig, ax = plt.subplots(figsize=(7, 3.5))
    m = kpi_aport["mensual"]
    ax.bar(m["periodo"], m["recaudo"] / 1e6, color="#1F4E9C")
    ax.set_title("Recaudo mensual de aportes (millones de $)")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout(); rutas.append(carpeta + "recaudo_mensual.png"); fig.savefig(rutas[-1], dpi=110); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    t = kpi_pqrs["por_tema"].sort_values("pct_a_tiempo")
    ax.barh(t["tema"], t["pct_a_tiempo"], color="#008542")
    ax.axvline(90, color="#B02A26", ls="--", label="meta 90 %")
    ax.set_xlim(0, 100); ax.set_title("% de PQRS respondidas dentro de 15 días hábiles, por tema"); ax.legend(loc="lower left")
    fig.tight_layout(); rutas.append(carpeta + "oportunidad_pqrs.png"); fig.savefig(rutas[-1], dpi=110); plt.close(fig)
    return rutas

In [ ]:
from openpyxl.drawing.image import Image as XLImage

def exportar_informe(kpi_pqrs, kpi_aport, mora, calidad):
    carpeta = CONFIG["CARPETA_SALIDA"]
    ruta = f"{carpeta}Informe_Direccion_{date.today():%Y%m%d}.xlsx"
    resumen = pd.DataFrame(list({**kpi_pqrs["kpis"], **kpi_aport["kpis"]}.items()), columns=["Indicador", "Valor"])
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        resumen.to_excel(xw, sheet_name="Resumen", index=False)
        kpi_pqrs["por_tipo"].to_excel(xw, sheet_name="PQRS", index=False)
        # ✏️ escribe kpi_pqrs["por_tema"] en la hoja "PQRS" debajo de la anterior (startrow=...)
        # ✏️ escribe kpi_aport["mensual"] en la hoja "Aportes"
        # ✏️ escribe kpi_aport["por_regional"] en la hoja "Aportes" (startcol=5)
        # ✏️ escribe mora en la hoja "Mora" y calidad en la hoja "Calidad"
        hoja = xw.sheets["Resumen"]
        hoja.column_dimensions["A"].width = 45
        hoja.column_dimensions["B"].width = 18
        for i, img in enumerate(graficos(kpi_pqrs, kpi_aport, carpeta)):
            hoja.add_image(XLImage(img), f"D{2 + i * 20}")
    log(f"Informe exportado: {ruta}")
    return ruta

In [ ]:
def main():
    log("=== Inicio del pipeline ===")
    afiliados_raw, pqrs, aportes_raw, catalogo = cargar_datos()           # 1. Ingestar
    afiliados = limpiar_afiliados(afiliados_raw, catalogo)               # 2. Procesar
    aportes = limpiar_aportes(aportes_raw)
    calidad = evaluar_calidad(afiliados, aportes, pqrs, catalogo)         # 3. Validar
    criticas = calidad[(calidad["#"].isin([6, 7])) & (calidad["% cumplimiento"] < 95)]
    if len(criticas):
        log("⚠️ ALERTA: reglas críticas por debajo del 95 %: " + ", ".join(criticas["regla"]))
    kpi_pqrs = indicadores_pqrs(pqrs)                                      # 4. Indicadores
    kpi_aport = indicadores_aportes(aportes, afiliados, catalogo)
    mora = seudonimizar(kpi_aport["mora"])                                 # 5. Proteger
    ruta = exportar_informe(kpi_pqrs, kpi_aport, mora, calidad)            # 6. Reportar
    log("=== Fin del pipeline ===")
    return ruta

ruta_informe = main()

In [ ]:
# Descargar el informe (solo funciona en Colab)
try:
    from google.colab import files
    files.download(ruta_informe)
except ImportError:
    print("Informe guardado en:", ruta_informe)

## 🎤 Prepara tu presentación (3 minutos)
1. **El informe:** abre el Excel y muestra la hoja *Resumen*.
2. **Tres hallazgos** para la Dirección (escríbelos abajo).
3. **Confianza:** ¿qué regla de calidad te preocupa más y qué harías al respecto?

✍️ *Hallazgos:*
1.
2.
3.

### ⭐ Bonus (+10)
- Agrega una **regla de calidad propia** (por ejemplo: fecha de afiliación posterior al nacimiento + 18 años).
- Calcula el % de oportunidad **por mes** de radicación y grafícalo.
- Cambia `log()` por el módulo `logging` y guarda la bitácora en `salidas/pipeline.log`.